## Homework 6 — Yellow Taxi November 2025
Prototype notebook for Spark batch processing. Covers data loading, repartitioning, and SQL queries.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("yellow_tripdata_2025-11") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

#### Load & Inspect Raw Data

In [ ]:
# load & inspect raw data
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
input_path = RAW_DIR / "yellow_tripdata_2025-11.parquet"

trips_df = spark.read.parquet(str(input_path))
trips_df.printSchema()
trips_df.show(5, truncate=False)

In [ ]:
total_records = trips_df.count()
print(f"Total records (Nov 2025): {total_records:,}")

#### Q2: Average Partition File Size
Repartition into 4 files, write to the Gold layer, then inspect output sizes via terminal:
```bash
ls -lh data/pq/yellow/2025/11/
```

In [ ]:
PQ_DIR = DATA_DIR / "pq" / "yellow" / "2025" / "11"

trips_df.repartition(4) \
    .write.parquet(str(PQ_DIR), mode="overwrite")
print(f"✓ Written to {PQ_DIR}")

**Result:** Each part file is ~24MB → Answer: **25MB**

#### Register Temp View for SQL Queries

In [ ]:
trips_df.createOrReplaceTempView("yellow_tripdata")
print("✓ Temp view 'yellow_tripdata' registered")

#### Q3: Total Trips on November 15, 2025
Count all trips where the pickup date falls on Nov 15.  
`to_date()` strips the timestamp so we can filter by date only.

In [ ]:
trip_count = spark.sql("""
    SELECT 
        COUNT(*) AS trip_count
    FROM 
        yellow_tripdata
    WHERE 
        to_date(tpep_pickup_datetime) = '2025-11-15'
""")

trip_count.show()

#### Q4: Longest Trip Duration (in hours)
Find the single longest trip by calculating the difference between pickup and dropoff timestamps.

**Approach:**
- Convert both timestamps to Unix seconds with `unix_timestamp()`
- Subtract pickup from dropoff to get duration in seconds
- Divide by 3600 to convert to hours

In [ ]:
longest_trip = spark.sql("""
    SELECT 
        ROUND(
            MAX(unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600
        , 2) AS max_duration_hours
    FROM 
        yellow_tripdata
""")

longest_trip.show()

**Result:** Longest trip was `90.65` hours → Answer: **90.6**

#### Q5: Spark UI Port

Spark's Web UI runs on **port 4040** by default.

Access it while a Spark session is active:
```
http://localhost:4040
```

The UI shows running jobs, stages, tasks, storage, and executor details — useful for debugging performance bottlenecks.

**Answer: 4040**

#### Q6: Least Frequent Pickup Zone
Join trip data with the zone lookup table to find the pickup zone with the fewest trips.

In [ ]:
# Load zone lookup
zones_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("../taxi_zone_lookup.csv")

zones_df.createOrReplaceTempView("zones")

print(f"✓ Zones loaded: {zones_df.count()} records")
zones_df.show(5, truncate=False)

In [ ]:
least_frequent_zones = spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(*) AS trip_count
    FROM 
        yellow_tripdata t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        trip_count ASC
    LIMIT 5
""")

least_frequent_zones.show(truncate=False)